# Supplementary Figure 3 - COSTE nested heatmaps

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** A100 nested COSTE benchmark outputs.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
# Panel mapping:
# - Supplementary Figure 3a-l, shared setup. Imports plotting and table libraries used by the COSTE nested heatmap panels.
#

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
sns.set_theme(context="talk", style="white")


In [ ]:
# Panel mapping:
# - Supplementary Figure 3a-l, data-generation helper. Recreates the twelve nested perturbation datasets used as COSTE inputs.
#

CELL_TYPES = list("ABCDEF")


def make_synthetic_modular(n_per_type=200, seed=42, spread=0.6, module12_dist=5.0):
    """Create the modular synthetic pattern: A/B, C/D, and E/F form three modules."""
    rng = np.random.default_rng(seed)
    centers = {
        "A": (0.0, 0.0),
        "B": (0.0, 0.0),
        "C": (module12_dist, 0.0),
        "D": (module12_dist, 0.0),
        "E": (module12_dist / 2.0, 5.0),
        "F": (module12_dist / 2.0, 5.0),
    }
    frames = []
    for label, (cx, cy) in centers.items():
        xy = rng.normal(loc=(cx, cy), scale=spread, size=(n_per_type, 2))
        frames.append(pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "celltype": label}))
    return pd.concat(frames, ignore_index=True)


def _annulus(rng, center, n, inner, outer):
    theta = rng.uniform(0, 2 * np.pi, n)
    radius = np.sqrt(rng.uniform(inner**2, outer**2, n))
    return np.column_stack([center[0] + radius * np.cos(theta), center[1] + radius * np.sin(theta)])


def make_nested_variant(
    n_core=150,
    n_middle=250,
    n_outer=400,
    center1=(0.0, 0.0),
    center2=(8.0, 8.0),
    radii=((0, 0.4), (0.5, 1.1), (1.2, 2.2)),
    seed=123,
):
    """Create nested A/B/C and D/E/F shell structures."""
    rng = np.random.default_rng(seed)
    specs = [
        ("A", center1, n_core, *radii[0]),
        ("B", center1, n_middle, *radii[1]),
        ("C", center1, n_outer, *radii[2]),
        ("D", center2, n_core, *radii[0]),
        ("E", center2, n_middle, *radii[1]),
        ("F", center2, n_outer, *radii[2]),
    ]
    frames = []
    for label, center, n, inner, outer in specs:
        xy = _annulus(rng, center, n, inner, outer)
        frames.append(pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "celltype": label}))
    return pd.concat(frames, ignore_index=True)


def make_synthetic_random(n_per_type=300, seed=42):
    """Create the well-mixed random negative-control pattern."""
    rng = np.random.default_rng(seed)
    frames = []
    for label in CELL_TYPES:
        xy = rng.uniform(-1, 9, size=(n_per_type, 2))
        frames.append(pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "celltype": label}))
    return pd.concat(frames, ignore_index=True)


def nested_dataset_collection():
    """Return the twelve nested perturbation datasets used in Supplementary Figure 2."""
    radius_conditions = [
        ((0, 0.3), (0.5, 1.0), (1.2, 2.0)),
        ((0, 0.4), (0.6, 1.2), (1.5, 2.6)),
        ((0, 0.5), (0.8, 1.6), (2.0, 3.2)),
        ((0, 0.7), (1.0, 2.0), (2.5, 4.0)),
    ]
    thickness_conditions = [
        ((0, 0.35), (0.45, 0.75), (0.85, 1.2)),
        ((0, 0.35), (0.55, 1.0), (1.2, 1.9)),
        ((0, 0.35), (0.65, 1.3), (1.6, 2.7)),
        ((0, 0.35), (0.80, 1.7), (2.1, 3.6)),
    ]
    center_conditions = [((0, 0), (5, 5)), ((0, 0), (7, 7)), ((0, 0), (9, 9)), ((0, 0), (11, 11))]
    out = {}
    for i, radii in enumerate(radius_conditions, 1):
        out[f"nested_radius_{i}"] = make_nested_variant(radii=radii, seed=100 + i)
    for i, radii in enumerate(thickness_conditions, 1):
        out[f"nested_thickness_{i}"] = make_nested_variant(radii=radii, seed=200 + i)
    for i, (center1, center2) in enumerate(center_conditions, 1):
        out[f"nested_centers_{i}"] = make_nested_variant(center1=center1, center2=center2, seed=300 + i)
    return out


In [ ]:
# Panel mapping:
# - Supplementary Figure 3a-l, plotting helper. Provides the heatmap styling for the nested COSTE panels.
#

def plot_spatial_pattern(df, title, output_pdf):
    """Save a spatial point-cloud panel with equal x/y scale."""
    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 6))
    sns.scatterplot(data=df, x="x", y="y", hue="celltype", s=8, linewidth=0, palette="tab10", ax=ax)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.legend(title="Cell type", frameon=False, markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(output_pdf, bbox_inches="tight")
    plt.close(fig)


def plot_matrix(matrix, title, output_pdf, cmap="RdBu_r", center=None, cbar_label="Score"):
    """Save a square heatmap used by the benchmark panels."""
    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(matrix, cmap=cmap, center=center, square=True, linewidths=0.4, cbar_kws={"label": cbar_label}, ax=ax)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_pdf, bbox_inches="tight")
    plt.close(fig)


In [ ]:
# Panel mapping:
# - Supplementary Figure 3a-l, Cell-GPS/COSTE helper. Computes the Searcher's D / SSS StructureMap matrix for each nested dataset.
#

from sfplot import compute_cophenetic_distances_from_df, plot_cophenetic_heatmap


def compute_coste_structuremap(points: pd.DataFrame, sample_name: str, output_dir: Path):
    """Compute the COSTE/Cell-GPS StructureMap and save the heatmap/table.

    The required table columns are `x`, `y`, and `celltype`. The manuscript uses the
    row cophenetic matrix as the Searcher's D / SSS heatmap.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    row_coph, col_coph = compute_cophenetic_distances_from_df(
        points[["x", "y", "celltype"]],
        x_col="x",
        y_col="y",
        celltype_col="celltype",
        method="average",
        show_corr=False,
    )
    row_coph.to_csv(output_dir / f"StructureMap_table_{sample_name}.csv")
    plot_cophenetic_heatmap(
        row_coph,
        matrix_name="row_coph",
        output_filename=str(output_dir / f"StructureMap_of_{sample_name}.pdf"),
        sample=sample_name,
    )
    return row_coph, col_coph


In [ ]:
# Panel mapping:
# - Supplementary Figure 3a-l, panel-generation cell. Writes one COSTE SSS heatmap per nested synthetic condition.
#

OUTPUT_DIR = Path("outputs/supp_fig_03_COSTE_nested_heatmaps")
for name, df in nested_dataset_collection().items():
    row_coph, _ = compute_coste_structuremap(df, name, OUTPUT_DIR)
    plot_matrix(row_coph, f"COSTE SSS - {name}", OUTPUT_DIR / f"COSTE_{name}.pdf", cbar_label="COSTE SSS")
